# AI for Actuaries : Hackathon 2026

## Week 1: Build your Pricing Approach

**Objective**

Predict the expected ultimate net claim cost (risk premium) for each policy-period in 2025, net of policy excess and recoveries, excluding repudiated claims.

The modelling approach is based on:

$ \text{Expected Claim Cost} = \text{Exposure} \times \text{Frequency} \times \text{Severity} $



In [1]:
# Install packages
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# EDA
from data_profiling import ProfileReport

# Models

warnings.filterwarnings('ignore')

import random
import os

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

/Users/sarahguilliams/anaconda3/envs/hackathon/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step-by-Step Analysis

### 1. Data Understanding
Load dataset

Show first few rows

Key observations

In [2]:
#Read the datasets
from pathlib import Path

data_dir = Path("data")
if not data_dir.is_dir():
    raise FileNotFoundError(f"Data directory not found: {data_dir.resolve()}")

parquet_files = sorted(data_dir.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No Parquet datasets found in {data_dir.resolve()}")

datasets = {path.stem: pd.read_parquet(path) for path in parquet_files}

display(pd.DataFrame(
    [
        {"dataset": name, "rows": len(df), "columns": len(df.columns), "duplicate rows": df.duplicated().sum()}
        for name, df in datasets.items()
    ]
))

for name, df in datasets.items():
    print(f"\nFirst 10 rows of {name}:")
    print(df.head(10).to_string(index=False))

,dataset,rows,columns,duplicate rows
0,addresses,575983,16,0
1,claims,163150,12,0
2,customers,545240,18,0
3,policies_buildings,707726,27,0
4,policies_contents,721617,28,0
5,policies_motor,979968,39,0
6,sample_submission,626639,2,0
7,test_buildings,184155,26,0
8,test_contents,187498,27,0
9,test_motor,254986,38,0



First 10 rows of addresses:
address_id region urban_density  flood_zone  distance_to_river_km  elevation_m soil_type  year_built property_type roof_type  basement  trees_within_5m  listed_building  bedrooms energy_rating  solar_panels
  A1980663    R05      suburban         1.0                  4.10          9.2      clay        2014      terraced   pitched     False            False            False         3             A         False
  A7251041    R12         urban         1.0                  0.39          4.0     chalk        1969      terraced   pitched     False            False            False         4             C         False
  A4921189    R11         rural         2.0                  0.41         60.0      clay        1952      terraced   pitched     False            False            False         3             C         False
  A9620763    R04         urban         1.0                  6.89          4.7      clay        1881          flat   pitched     False         

### 2. Data Preparation

Objective: Ensure that each row represents a valid policy-period and that reference tables have unique keys. \
Below the table we can see that the average exposure in the policies is more than 1 year.

In [3]:
customers = datasets["customers"].copy()
addresses = datasets["addresses"].copy()
weather = datasets["weather"].copy()
claims = datasets["claims"].copy()

train = {
    line: datasets[f"policies_{line}"].copy()
    for line in ["motor", "buildings", "contents"]
}

test = {
    line: datasets[f"test_{line}"].copy()
    for line in ["motor", "buildings", "contents"]
}

# 1. Check unique keys
print("=== REFERENCE TABLE CHECKS ===")

for name, df, keys in [
    ("customers", customers, ["customer_id"]),
    ("addresses", addresses, ["address_id"]),
    ("weather", weather, ["address_id", "year"]),
    ("claims", claims, ["claim_id"])
]:
    duplicates = df.duplicated(subset=keys).sum()
    status = "PASS" if duplicates == 0 else "FAIL"
    print(f"{name:20} | {status} | Duplicate keys: {duplicates}")

# 2. Check policy-period structure
print("\n=== POLICY-PERIOD CHECKS ===")

results = []

for line in ["motor", "buildings", "contents"]:
    for dataset_type, df in [
        ("train", train[line]),
        ("test", test[line])
    ]:
        df["period_start"] = pd.to_datetime(df["period_start"])
        df["period_end"] = pd.to_datetime(df["period_end"])

        results.append({
            "dataset": f"{dataset_type}_{line}",
            "rows": len(df),
            "duplicate_row_ids": df["row_id"].duplicated().sum(),
            "invalid_dates": (
                df["period_start"] > df["period_end"]
            ).sum(),
            "missing_exposure": df["exposure"].isna().sum(),
            "nonpositive_exposure": (df["exposure"] <= 0).sum(),
            "total_exposure": df["exposure"].sum(),
            "average_exposure": df["exposure"].mean()
        })

quality_report = pd.DataFrame(results)

display(quality_report)

=== REFERENCE TABLE CHECKS ===
customers            | PASS | Duplicate keys: 0
addresses            | PASS | Duplicate keys: 0
weather              | PASS | Duplicate keys: 0
claims               | PASS | Duplicate keys: 0

=== POLICY-PERIOD CHECKS ===


,dataset,rows,duplicate_row_ids,invalid_dates,missing_exposure,nonpositive_exposure,total_exposure,average_exposure
0,train_motor,979968,0,0,0,0,1.058510e+06,1.080147
1,test_motor,254986,0,0,0,0,1.999836e+05,0.784292
2,train_buildings,707726,0,0,0,0,7.894039e+05,1.115409
3,test_buildings,184155,0,0,0,0,1.502439e+05,0.815856
4,train_contents,721617,0,0,0,0,8.107226e+05,1.123480
5,test_contents,187498,0,0,0,0,1.501238e+05,0.800669


From performing these checks we notice that the exposure has a value that is larger than 1. This is not possible as the description of the dataset indicated that each line in the policy datasets in one calender year for each policy.
Checking the exposures indicates that there are some inconsistencies in the data. 

In [4]:
# Check the exposures:
for line in ["motor", "buildings", "contents"]:

    df = datasets[f"policies_{line}"]

    print(f"\n=== {line.upper()} ===")

    print(df["exposure"].describe(
        percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]
    ))

    print("Rows with exposure > 1:",
          (df["exposure"] > 1).sum())

    print("Maximum exposure:", df["exposure"].max())

for line in ["motor", "buildings", "contents"]:

    df = datasets[f"policies_{line}"]

    exposure_above_1 = df[df["exposure"] > 1]

    print(f"\n=== {line.upper()} ===")
    print(f"Number of rows with exposure > 1: {len(exposure_above_1)}")

    #display(exposure_above_1)


=== MOTOR ===
count    979968.000000
mean          1.080147
std           9.635858
min           0.002700
1%            0.021900
25%           0.638400
50%           1.000000
75%           1.000000
95%           1.000000
99%           1.000000
max         365.000000
Name: exposure, dtype: float64
Rows with exposure > 1: 3991
Maximum exposure: 365.0

=== BUILDINGS ===
count    707726.000000
mean          1.115409
std           9.779618
min           0.002700
1%            0.027400
25%           0.764400
50%           1.000000
75%           1.000000
95%           1.000000
99%           1.000000
max         365.000000
Name: exposure, dtype: float64
Rows with exposure > 1: 3044
Maximum exposure: 365.0

=== CONTENTS ===
count    721617.00000
mean          1.12348
std          10.16077
min           0.00270
1%            0.02470
25%           0.69860
50%           1.00000
75%           1.00000
95%           1.00000
99%           1.00000
max         365.00000
Name: exposure, dtype: float64
R

In [5]:
# Recalculate exposure quality report

results = []

for line in ["motor", "buildings", "contents"]:
    for dataset_type, df in [
        ("train", train[line]),
        ("test", test[line])
    ]:

        df["period_start"] = pd.to_datetime(df["period_start"])
        df["period_end"] = pd.to_datetime(df["period_end"])

        # Recalculate exposure based on actual days covered
        days_covered = (
            df["period_end"] - df["period_start"]
        ).dt.days + 1

        # Account for leap years
        days_in_year = np.where(
            df["period_start"].dt.is_leap_year,
            366,
            365
        )

        df["exposure_corrected"] = days_covered / days_in_year

        # Build quality report using corrected exposure
        results.append({
            "dataset": f"{dataset_type}_{line}",
            "rows": len(df),
            "duplicate_row_ids": df["row_id"].duplicated().sum(),
            "invalid_dates": (
                df["period_start"] > df["period_end"]
            ).sum(),
            "missing_exposure": df["exposure_corrected"].isna().sum(),
            "nonpositive_exposure": (
                df["exposure_corrected"] <= 0
            ).sum(),
            "total_exposure": df["exposure_corrected"].sum(),
            "average_exposure": df["exposure_corrected"].mean(),
            "max_exposure": df["exposure_corrected"].max(),
            "original_average_exposure": df["exposure"].mean()
        })

quality_report = pd.DataFrame(results)

display(quality_report.round(4))

,dataset,rows,duplicate_row_ids,invalid_dates,missing_exposure,nonpositive_exposure,total_exposure,average_exposure,max_exposure,original_average_exposure
0,train_motor,979968,0,0,0,0,786611.6609,0.8027,1.0,1.0801
1,test_motor,254986,0,0,0,0,199983.5808,0.7843,1.0,0.7843
2,train_buildings,707726,0,0,0,0,590150.5701,0.8339,1.0,1.1154
3,test_buildings,184155,0,0,0,0,150243.9041,0.8159,1.0,0.8159
4,train_contents,721617,0,0,0,0,590449.5397,0.8182,1.0,1.1235
5,test_contents,187498,0,0,0,0,150123.7699,0.8007,1.0,0.8007


In [6]:
# After performing the exposure correction, we will update the original exposure column with the corrected values in both train and test datasets.
for line in train:
    train[line]["exposure"] = train[line]["exposure_corrected"]

for line in test:
    test[line]["exposure"] = test[line]["exposure_corrected"]


Merge customer and adress info.
Objective: Enrich policy-level features before building the target.
These are many-to-one joins: each policy-period belongs to one customer and references one insured address.

In [7]:
def enrich_static_features(df, line, dataset_name):
    initial_rows = len(df)

    enriched = df.merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one"
    )

    enriched = enriched.merge(
        addresses,
        on="address_id",
        how="left",
        validate="many_to_one"
    )

    enriched_rows = len(enriched)
    if enriched_rows != initial_rows:
        print(
            f"{dataset_name} {line}: FAIL — row count changed from "
            f"{initial_rows} to {enriched_rows}"
        )
        raise ValueError(
            f"Row count changed during customer/address enrichment "
            f"for {dataset_name} {line}"
        )

    print(f"{dataset_name} {line}: PASS — row count unchanged ({initial_rows})")

    return enriched


print("Check AI code:")

train = {
    line: enrich_static_features(df, line, "Train")
    for line, df in train.items()
}

test = {
    line: enrich_static_features(df, line, "Test")
    for line, df in test.items()
}

Check AI code:
Train motor: PASS — row count unchanged (979968)
Train buildings: PASS — row count unchanged (707726)
Train contents: PASS — row count unchanged (721617)
Test motor: PASS — row count unchanged (254986)
Test buildings: PASS — row count unchanged (184155)
Test contents: PASS — row count unchanged (187498)


#### Step 4 — Engineer historical weather features
Objective: Add weather risk information without using future observations.
We should not use actual weather from 2025 to predict 2025 risk premium. Equally, for a 2023 policy we should not use actual weather outcomes from 2023 when simulating a prediction made at inception.
A starting approach is to use previous-year weather.

In [8]:
# Identify weather variables
weather_columns = [
    col for col in weather.columns
    if col not in ["address_id", "year"]
]

# Rename weather columns to indicate lagged features
weather_lagged = weather.rename(
    columns={
        "year": "previous_year",
        **{
            col: f"lag1_{col}"
            for col in weather_columns
        }
    }
)

# Check uniqueness of weather records
assert not weather_lagged.duplicated(
    subset=["address_id", "previous_year"]
).any(), "Duplicate address-year combinations in weather data"


# Function to merge historical weather
def merge_weather(df, line, dataset_name):

    result = df.copy()

    # Derive year from the policy-period start date
    result["period_start"] = pd.to_datetime(
        result["period_start"]
    )

    result["year"] = result["period_start"].dt.year

    # Use 2021 weather for 2021 policies,
    # previous-year weather for all subsequent years
    result["previous_year"] = np.where(
        result["year"] == 2021,
        2021,
        result["year"] - 1
    )

    original_rows = len(result)

    # Merge weather
    result = result.merge(
        weather_lagged,
        on=["address_id", "previous_year"],
        how="left",
        validate="many_to_one"
    )

    # Check row count
    merged_rows = len(result)

    if merged_rows != original_rows:
        print(
            f"{dataset_name} {line}: FAIL — row count changed from "
            f"{original_rows} to {merged_rows}"
        )
        raise ValueError(
            f"Row count changed during weather enrichment "
            f"for {dataset_name} {line}"
        )

    print(
        f"{dataset_name} {line}: PASS — "
        f"row count unchanged ({original_rows:,})"
    )

    return result.drop(columns="previous_year")


# For now also add weather data to motor
print("Check weather enrichment:")

# Apply to training datasets
train = {
    line: merge_weather(df, line, "Train")
    for line, df in train.items()
}

# Apply to test datasets
test = {
    line: merge_weather(df, line, "Test")
    for line, df in test.items()
}

Check weather enrichment:
Train motor: PASS — row count unchanged (979,968)
Train buildings: PASS — row count unchanged (707,726)
Train contents: PASS — row count unchanged (721,617)
Test motor: PASS — row count unchanged (254,986)
Test buildings: PASS — row count unchanged (184,155)
Test contents: PASS — row count unchanged (187,498)


#### Step 5 — Match claims to the correct policy-period
Objective: Assign every historical claim to the exact row covering its loss date.
This is the most important merge in the workflow.
We cannot join claims only on policy_id, because a policy may have multiple rows across different years or mid-term changes. So we do not first perform aggregation on the claims table with policy_id and loss year because there might be multiple policy versions within a year.


First, combine the policy-period identifiers into a temporary lookup table. This includes the policy_id and the start and end date of that version of the policy.

At the end there are some claims that are unmatched. As an actuary, it is necesarry to perform a check on the data to understand why this occurs. A diagnostic indicates that the loss date occured before the start date of the policy. In the majority of the cases, the reporting date did occur when a policy was risk-bearing. Therefore, the decision is made to link the "unmatched" cases to the policy such that loss date is nearest to the policy instance.

In [9]:
policy_windows = pd.concat(
    [
        datasets[f"policies_{line}"][
            ["row_id", "policy_id", "period_start", "period_end"]
        ].assign(line=line)
        for line in ["motor", "buildings", "contents"]
    ],
    ignore_index=True
)

policy_windows["period_start"] = pd.to_datetime(
    policy_windows["period_start"]
)
policy_windows["period_end"] = pd.to_datetime(
    policy_windows["period_end"]
)


## Exclude repudiated claims, then match the remaining claims to policy periods.
claims_for_matching = claims.loc[
    claims["status"] != "repudiated"
].copy()
claims_for_matching["loss_date"] = pd.to_datetime(
    claims_for_matching["loss_date"]
)

claims_with_policy_windows = claims_for_matching.merge(
    policy_windows,
    on=["policy_id", "line"],
    how="left"
)

claims_matched = claims_with_policy_windows.loc[
    (claims_with_policy_windows["loss_date"] >= claims_with_policy_windows["period_start"])
    &
    (claims_with_policy_windows["loss_date"] <= claims_with_policy_windows["period_end"])
].copy()

print("Claims excluded (repudiated):", len(claims) - len(claims_for_matching))
print("Claims included in matching:", len(claims_for_matching))
print("Matched claims:", len(claims_matched))

print(
    "Claims with multiple matches:",
    claims_matched["claim_id"].duplicated().sum()
)

assert claims_matched["claim_id"].is_unique

unmatched_claims = claims_for_matching.loc[
    ~claims_for_matching["claim_id"].isin(claims_matched["claim_id"])
].copy()

unmatched_policy_windows = unmatched_claims.merge(
    policy_windows,
    on=["policy_id", "line"],
    how="left",
    indicator=True
)

policy_window_summary = (
    unmatched_policy_windows.groupby("claim_id")
    .agg(
        policy_found=("_merge", lambda matches: matches.eq("both").any()),
        first_period_start=("period_start", "min"),
        last_period_end=("period_end", "max")
    )
    .reset_index()
)

unmatched_claim_diagnostics = unmatched_claims[
    ["claim_id", "policy_id", "line", "loss_date",  "report_date"]
].merge(policy_window_summary, on="claim_id", how="left")

reasons = [
    unmatched_claim_diagnostics["loss_date"].isna(),
    ~unmatched_claim_diagnostics["policy_found"],
    unmatched_claim_diagnostics["first_period_start"].isna()
    | unmatched_claim_diagnostics["last_period_end"].isna(),
    unmatched_claim_diagnostics["loss_date"]
    < unmatched_claim_diagnostics["first_period_start"],
    unmatched_claim_diagnostics["loss_date"]
    > unmatched_claim_diagnostics["last_period_end"]
]
reason_labels = [
    "Missing loss date",
    "No matching policy_id and line",
    "Missing policy period dates",
    "Loss before first policy period",
    "Loss after last policy period"
]
unmatched_claim_diagnostics["reason"] = np.select(
    reasons,
    reason_labels,
    default="Gap between policy periods"
)

print("Check AI code:")
print("Unmatched claims:", len(unmatched_claim_diagnostics))
print("Unmatched claim reasons:")
print(unmatched_claim_diagnostics["reason"].value_counts().to_string())



# Find candidate periods for claims that did not match by loss date
nearest_candidates = unmatched_claims.merge(
    policy_windows,
    on=["policy_id", "line"],
    how="inner",
    validate="many_to_many"
)

# Distance is zero if the date is inside the interval; otherwise, use
# the number of days to the nearest period boundary.
nearest_candidates["distance_days"] = np.select(
    [
        nearest_candidates["loss_date"] < nearest_candidates["period_start"],
        nearest_candidates["loss_date"] > nearest_candidates["period_end"],
    ],
    [
        (nearest_candidates["period_start"] - nearest_candidates["loss_date"]).dt.days,
        (nearest_candidates["loss_date"] - nearest_candidates["period_end"]).dt.days,
    ],
    default=0
)

# Select one nearest policy-period for each claim.
nearest_matches = (
    nearest_candidates
    .dropna(subset=["loss_date", "period_start", "period_end"])
    .sort_values(["claim_id", "distance_days", "period_start", "row_id"])
    .drop_duplicates("claim_id", keep="first")
    .copy()
)

nearest_matches["match_method"] = "nearest_period"
claims_matched["match_method"] = "loss_date_within_period"

# Add nearest-period assignments to the existing exact matches.
claims_matched = pd.concat(
    [claims_matched, nearest_matches],
    ignore_index=True
)

assert claims_matched["claim_id"].is_unique

print("Claims assigned to nearest period:", len(nearest_matches))
print("Claims still without a policy period:",
      len(unmatched_claims) - len(nearest_matches))

Claims excluded (repudiated): 1711
Claims included in matching: 161439
Matched claims: 161029
Claims with multiple matches: 0
Check AI code:
Unmatched claims: 410
Unmatched claim reasons:
reason
Loss before first policy period    410
Claims assigned to nearest period: 410
Claims still without a policy period: 0


#### Step 6 — Calculate the observed net incurred claim cost
Objective: Construct a historical cost measure consistent with the target.
According to the data dictionary:
- paid_to_date is already net of excess.
- incurred_to_date consists of paid amounts and the open-claim case estimate.
- recoveries is a separate amount, but it is not confirmed whether incurred already reflects it.
- Repudiated claims must be excluded.

Our provisional assumption is:
$ \text{Observed Net Incurred} = \text{Incurred to Date} - \text{Recoveries} $

We must not deduct the policy excess again.

In [10]:
claims_matched["recoveries"] = (
    claims_matched["recoveries"].fillna(0)
)

# Working assumption: recoveries are NOT yet deducted from incurred_to_date.
RECOVERIES_ALREADY_NETTED = False

if RECOVERIES_ALREADY_NETTED:
    claims_matched["net_cost_before_status"] = (
        claims_matched["incurred_to_date"]
    )
else:
    claims_matched["net_cost_before_status"] = (
        claims_matched["incurred_to_date"]
        - claims_matched["recoveries"]
    )

claims_matched["net_incurred"] = np.where(
    claims_matched["net_cost_before_status"] > 0,
    claims_matched["net_cost_before_status"],
    0
)

# Final reconciliation:
display(
    claims_matched.groupby(["line", "status"]).agg(
        claims=("claim_id", "count"),
        incurred=("incurred_to_date", "sum"),
        recoveries=("recoveries", "sum"),
        net_incurred=("net_incurred", "sum")
    )
)

negative_claims = claims_matched[
    claims_matched["net_incurred"] < 0
]

print("Negative net incurred claims:", len(negative_claims))

claims      incurred   recoveries  net_incurred
line      status                                                 
buildings closed   24549  2.169569e+08         0.00  2.169569e+08
          open      1893  3.124254e+07         0.00  3.124254e+07
contents  closed   22992  6.265879e+07         0.00  6.265879e+07
          open       810  3.470007e+06         0.00  3.470007e+06
motor     closed  103751  3.773012e+08  27678633.34  3.496226e+08
          open      7444  8.210164e+07         0.00  8.210164e+07

Negative net incurred claims: 0


#### Step 7 — Aggregate claims at policy-period level, and per policy level generate the target values:
- $N_{i}$ : number of claims per policy, given that exposure =1.
- the average claims cost, given that N > 0.

Objective: Construct the two historical responses: N and total observed claim cost.


In [11]:
# 1. Define targets
targets = (
    claims_matched
    .groupby("row_id", as_index=False)
    .agg(
        nclaims=("claim_id", "nunique"),
        net_incurred=("net_incurred", "sum")
    )
)

targets["nclaims"] = targets["nclaims"].astype(int)


# 2. Merge targets into each line of business
for line, df in train.items():

    initial_rows = len(df)

    # Remove target columns if cell is rerun
    df = df.drop(
        columns=["nclaims", "net_incurred", "avg_sev"],
        errors="ignore"
    )

    # Merge targets
    df = df.merge(
        targets,
        on="row_id",
        how="left",
        validate="one_to_one"
    )

    # Fill missing values
    df["nclaims"] = df["nclaims"].fillna(0).astype(int)
    df["net_incurred"] = df["net_incurred"].fillna(0)

    # Average severity: 0 when nclaims = 0
    df["avg_sev"] = (
        df["net_incurred"]
        / df["nclaims"].replace(0, np.nan)
    ).fillna(0)


    # Validate
    assert len(df) == initial_rows, (
        f"{line}: Row count changed from "
        f"{initial_rows} to {len(df)}"
    )

    train[line] = df

    print(
        f"{line}: {len(df):,} rows preserved | "
        f"{df['nclaims'].sum():,} claims | "
        f"{df['net_incurred'].sum():,.2f} net incurred"
    )

motor: 979,968 rows preserved | 111,195 claims | 431,724,224.25 net incurred
buildings: 707,726 rows preserved | 26,442 claims | 248,199,474.78 net incurred
contents: 721,617 rows preserved | 23,802 claims | 66,128,796.73 net incurred


#### Step 8 — Assess claim development to ultimate
Objective: Correct the mismatch between observed incurred and ultimate costs.
At 31 December 2024:
- Some 2024 losses are not yet reported (IBNR).
- Some reported claims remain open and may develop.
- Future recoveries may not yet be reflected.
Therefore, modelling observed_net_incurred directly will generally underestimate ultimate loss costs if those effects are material.
Ideally, use historical valuation snapshots to build paid/incurred development triangles. However, the supplied claims table has only one snapshot, so reliable chain-ladder development factors cannot be estimated from this table alone.
If validated development-to-ultimate factors become available, you can apply them at a suitable line/peril/accident-year level.

In [12]:
# No estimated development factors are supplied yet.
# These should NOT be treated as genuine ultimate estimates.

for line in train:
    train[line]["development_factor"] = 1.0

    train[line]["avg_sev"] = (
        train[line]["avg_sev"]
        * train[line]["development_factor"]
    )

Here I want a visualization of how the datasets are cleaned from before , to after and the variables that were added

In [13]:
lines_for_comparison = {
    "Motor": "motor",
    "Buildings": "buildings",
    "Contents": "contents",
}

target_columns = ["nclaims", "avg_sev"]

# Summarize rows, exposure, target availability, and dtype counts.
comparison_rows = []
schema_rows = []

for label, line in lines_for_comparison.items():
    before = datasets[f"policies_{line}"]
    after = train[line]

    comparison_rows.extend([
        {
            "line": label,
            "stage": "Before",
            "rows": len(before),
            "columns": len(before.columns),
            "mean exposure": before["exposure"].mean(),
            "total exposure": before["exposure"].sum(),
            "targets present": "None",
        },
        {
            "line": label,
            "stage": "After",
            "rows": len(after),
            "columns": len(after.columns),
            "mean exposure": after["exposure"].mean(),
            "total exposure": after["exposure"].sum(),
            "targets present": ", ".join(
                col for col in target_columns if col in after.columns
            ),
        },
    ])

    before_cols = set(before.columns)
    after_cols = set(after.columns)

    for col in sorted(before_cols | after_cols):
        schema_rows.append({
            "line": label,
            "column": col,
            "before dtype": str(before[col].dtype) if col in before_cols else "—",
            "after dtype": str(after[col].dtype) if col in after_cols else "—",
            "change": (
                "Added" if col not in before_cols else
                "Removed" if col not in after_cols else
                "Retained"
            ),
        })

comparison_summary = pd.DataFrame(comparison_rows)
schema_comparison = pd.DataFrame(schema_rows)

display(comparison_summary.round(3))
#display(schema_comparison)


,line,stage,rows,columns,mean exposure,total exposure,targets present
0,Motor,Before,979968,39,1.080,1058509.681,None
1,Motor,After,979968,84,0.803,786611.661,"nclaims, avg_sev"
2,Buildings,Before,707726,27,1.115,789403.927,None
3,Buildings,After,707726,72,0.834,590150.570,"nclaims, avg_sev"
4,Contents,Before,721617,28,1.123,810722.569,None
5,Contents,After,721617,73,0.818,590449.540,"nclaims, avg_sev"


In [14]:
print(train["motor"].head(10).to_string())

      row_id  policy_id customer_id address_id period_start period_end  exposure  driver_age  years_licensed  ncb_years  vehicle_value_group  vehicle_age make_model  engine_size_cc  body_type transmission  seats vehicle_colour fuel_type  driver_assist     cover_type  annual_mileage overnight_parking vehicle_use  named_young_driver  telematics_opt_in  telematics_score  voluntary_excess  compulsory_excess          channel payment_method  missed_payments_12m  new_business  renewal_count  days_quote_to_inception  discount_applied  written_premium  customer_service_contacts renewal_decision  exposure_corrected     surname date_of_birth   occupation marital_status  household_size credit_band  prior_claims_5y  licence_points  years_at_address  customer_since_year email_domain marketing_consent document_preference previous_insurer campaign_code  branch_id  marketing_segment region urban_density  flood_zone  distance_to_river_km  elevation_m soil_type  year_built property_type roof_type  baseme

### 3. EDA

In [15]:
# Directory for HTML reports
report_dir = Path("profile_reports")
report_dir.mkdir(parents=True, exist_ok=True)

# Datasets to profile
profile_datasets = {
    "policies_motor": datasets["policies_motor"],
    "policies_buildings": datasets["policies_buildings"],
    "policies_contents": datasets["policies_contents"],
    "addresses": datasets["addresses"],
    "customers": datasets["customers"],
}

# Generate and save reports
for name, df in profile_datasets.items():
    print(f"Generating profile report: {name}")

    profile = ProfileReport(
        df,
        title=f"Profiling Report - {name}",
        explorative=True,
        minimal=True
    )

    output_path = report_dir / f"{name}_profile.html"
    profile.to_file(output_path)

    print(f"Saved: {output_path}")

print("All profile reports generated successfully.")

Generating profile report: policies_motor


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 156.45it/s]


Saved: profile_reports/policies_motor_profile.html
Generating profile report: policies_buildings


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 236.22it/s]


Saved: profile_reports/policies_buildings_profile.html
Generating profile report: policies_contents


...

Saved: profile_reports/policies_contents_profile.html
Generating profile report: addresses


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 410.68it/s]


Saved: profile_reports/addresses_profile.html
Generating profile report: customers


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 137.14it/s]

Saved: profile_reports/customers_profile.html
All profile reports generated successfully.


In [16]:
for label, line in lines_for_comparison.items():
    df = train[line]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    sns.histplot(
        data=df,
        x="nclaims",
        discrete=True,
        stat="probability",
        ax=axes[0]
    )
    axes[0].set(
        title="Claim count",
        xlabel="Number of claims",
        ylabel="Relative frequency"
    )

    sns.histplot(
        data=df,
        x="exposure",
        bins=30,
        stat="probability",
        ax=axes[1]
    )
    axes[1].set(
        title="Exposure",
        xlabel="Exposure",
        ylabel="Relative frequency"
    )

    positive_severity = df.loc[df["avg_sev"] > 0, "avg_sev"]
    sns.histplot(
        x=positive_severity,
        bins=40,
        stat="probability",
        log_scale=True,
        ax=axes[2]
    )
    axes[2].set(
        title="Average severity (claim periods)",
        xlabel="Average severity (log scale)",
        ylabel="Relative frequency"
    )

    fig.suptitle(f"{label}: Relative frequency distributions")
    plt.tight_layout()
    plt.show()

In [17]:
severity_data = pd.concat(
    [df.assign(line=line) for line, df in train.items()],
    ignore_index=True
)

plot_data = severity_data.loc[
    (severity_data["nclaims"] > 0) & severity_data["avg_sev"].notna()
].copy()

severity_comparison = (
    plot_data.groupby("line")["avg_sev"]
    .agg(
        count="count",
        median="median",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        max="max",
    )
)

severity_comparison["IQR"] = (
    severity_comparison["p75"]
    - plot_data.groupby("line")["avg_sev"].quantile(0.25)
)
severity_comparison["upper_fence"] = (
    severity_comparison["p75"] + 1.5 * severity_comparison["IQR"]
)

outlier_counts = (
    plot_data.groupby("line")
    .apply(lambda group: (group["avg_sev"] > severity_comparison.loc[group.name, "upper_fence"]).sum())
)
severity_comparison["outliers_above_fence"] = outlier_counts

display(severity_comparison.round(2))

,count,median,p75,p90,p95,p99,max,IQR,upper_fence,outliers_above_fence
line,,,,,,,,,,
buildings,25426,3160.42,8180.04,21071.28,38933.59,98222.80,631246.21,7050.49,18755.78,2857
contents,22963,1205.93,3211.80,6853.96,10500.49,21758.24,111900.00,2772.85,7371.07,2051
motor,98326,1627.88,4008.53,8230.94,12803.06,34276.06,11894276.93,3445.29,9176.46,8348


This is particularly useful for identifying whether the differences are in attritional claims or large losses.

### 4. Train test split.

In [18]:
# Define training and validation years
TRAIN_YEARS = [2021, 2022, 2023]
VALIDATION_YEAR = 2024

# Keep separate dictionaries for training and validation
train_split = {}
valid_split = {}

for line, df in train.items():

    df = df.copy()
    df["year"] = pd.to_datetime(
        df["period_start"]
    ).dt.year

    train_split[line] = df[
        df["year"].isin(TRAIN_YEARS)
    ].copy()

    valid_split[line] = df[
        df["year"] == VALIDATION_YEAR
    ].copy()

    print(f"\n=== {line.upper()} ===")
    print(f"Training rows:   {len(train_split[line]):,}")
    print(f"Validation rows: {len(valid_split[line]):,}")

    assert set(train_split[line]["row_id"]).isdisjoint(
        set(valid_split[line]["row_id"])
    )


=== MOTOR ===
Training rows:   725,270
Validation rows: 254,698

=== BUILDINGS ===
Training rows:   523,725
Validation rows: 184,001

=== CONTENTS ===
Training rows:   534,126
Validation rows: 187,491


In [19]:
def portfolio_summary(df):
    exposure = df["exposure"].sum()
    claims = df["nclaims"].sum()
    incurred = df["net_incurred"].sum()

    return pd.Series({
        "rows": len(df),
        "exposure": exposure,
        "claims": claims,
        "frequency": claims / exposure,
        "avg_severity": (
            incurred / claims if claims > 0 else np.nan
        ),
        "pure_premium": incurred / exposure
    })


split_summary = pd.concat(
    {
        line: pd.DataFrame({
            "train": portfolio_summary(train_split[line]),
            "validation": portfolio_summary(valid_split[line])
        }).T
        for line in train
    },
    names=["line", "split"]
)

display(split_summary.round(3))

rows    exposure   claims  frequency  avg_severity  \
line      split                                                                
motor     train       725270.0  587371.893  85316.0      0.145      3996.818   
          validation  254698.0  199239.768  25879.0      0.130      3505.998   
buildings train       523725.0  440642.658  19865.0      0.045      9097.815   
          validation  184001.0  149507.913   6577.0      0.044     10258.685   
contents  train       534126.0  440892.521  17810.0      0.040      2728.567   
          validation  187491.0  149557.019   5992.0      0.040      2926.071   

                      pure_premium  
line      split                     
motor     train            580.539  
          validation       455.390  
buildings train            410.147  
          validation       451.290  
contents  train            110.221  
          validation       117.233

For this first draft, use a small set of relatively stable underwriting features rather than every available variable.
This reduces the risk of leakage and lets you establish whether the model works before adding complexity.

### 5. Training the model

This is the piece of the code that can easily be adapted. Add new models here.

In [20]:
import numpy as np
import pandas as pd
from itertools import product

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import PoissonRegressor, GammaRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_poisson_deviance,
    mean_gamma_deviance,
    mean_tweedie_deviance,
    mean_absolute_error
)

SEED = 42

LINES = list(train_split)
EXPOSURE = "exposure"
FREQ_TARGET = "nclaims"
SEV_TARGET = "avg_sev"
COST_TARGET = "net_incurred"
CAP_QUANTILE = 0.995


# Add or remove algorithms here.
MODEL_GRIDS = {
    "frequency": {
        "glm": {
            "alpha": [0.001, 0.01, 0.1, 1.0]
        },
        "tree": {
            "max_depth": [3, 5],
            "min_samples_leaf": [500, 2000]
        },
        "xgboost": {
            "max_depth": [3, 5],
            "learning_rate": [0.05],
            "n_estimators": [300],
            "subsample": [0.8]
        }
    },
    "severity": {
        "glm": {
            "alpha": [0.001, 0.01, 0.1, 1.0]
        },
        "tree": {
            "max_depth": [3, 5],
            "min_samples_leaf": [100, 500]
        },
        "xgboost": {
            "max_depth": [3, 5],
            "learning_rate": [0.05],
            "n_estimators": [300],
            "subsample": [0.8]
        }
    }
}

# For the first run, compare only GLMs and trees.
# Add "xgboost" when ready.
ACTIVE_MODELS = {
    "frequency": ["xgboost"],
    "severity": [ "xgboost"]
}



# Columns that should never be predictors
exclude = {
    "row_id",
    "policy_id",
    "customer_id",
    "address_id",
    "period_start",
    "period_end",
    "year",
    "exposure",
    "exposure_original",
    "exposure_corrected",
    "exposure_difference",
    "nclaims",
    "avg_sev",
    "avg_claim",
    "net_incurred",
    "positive_incurred",
    "nclaims_positive",
    "avg_sev_positive",
    "development_factor", 
    "renewal_decision", #This is not included in the test set
    "surname",
    "prior_claims_5y",
    "years_at_address", "licence_points", "customer_since_year", "written_premium",
    "customer_service_contacts"
}

features = {
    line: [
        col for col in df.columns
        if col not in exclude
        and (
            pd.api.types.is_numeric_dtype(df[col])
            or pd.api.types.is_object_dtype(df[col])
            or isinstance(df[col].dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(df[col])
        )
    ]
    for line, df in train_split.items()
}

for line, cols in features.items():
    print("Features included in the model:")
    print(f"\n{line}: {len(cols)} features")
    print(cols)

Features included in the model:

motor: 63 features
['driver_age', 'years_licensed', 'ncb_years', 'vehicle_value_group', 'vehicle_age', 'make_model', 'engine_size_cc', 'body_type', 'transmission', 'seats', 'vehicle_colour', 'fuel_type', 'driver_assist', 'cover_type', 'annual_mileage', 'overnight_parking', 'vehicle_use', 'named_young_driver', 'telematics_opt_in', 'telematics_score', 'voluntary_excess', 'compulsory_excess', 'channel', 'payment_method', 'missed_payments_12m', 'new_business', 'renewal_count', 'days_quote_to_inception', 'discount_applied', 'date_of_birth', 'occupation', 'marital_status', 'household_size', 'credit_band', 'email_domain', 'marketing_consent', 'document_preference', 'previous_insurer', 'campaign_code', 'branch_id', 'marketing_segment', 'region', 'urban_density', 'flood_zone', 'distance_to_river_km', 'elevation_m', 'soil_type', 'year_built', 'property_type', 'roof_type', 'basement', 'trees_within_5m', 'listed_building', 'bedrooms', 'energy_rating', 'solar_panels

2. Data preparation and validation checks
Frequency uses all policy-periods. Severity uses only rows with strictly positive average claim cost.
We don't need separate permanent train_freq and train_sev dictionaries: the filtering can happen inside one training function.

In [21]:
def severity_filter(df):
    return df.loc[
        (df[SEV_TARGET] > 0)
    ].copy()


def check_data(df):
    assert (df[EXPOSURE] > 0).all()
    assert df[FREQ_TARGET].notna().all()
    assert (df[FREQ_TARGET] >= 0).all()
    assert np.isfinite(df[COST_TARGET]).all()
    assert (df[COST_TARGET] >= 0).all()


for line in LINES:
    check_data(train_split[line])
    check_data(valid_split[line])

    forbidden = {
        "row_id", "policy_id", "customer_id",
        "nclaims", "avg_sev", "net_incurred", "avg_claim", "development_factor"
    }

    assert not (set(features[line]) & forbidden), (
        f"{line}: target or identifier leakage in features"
    )

3. Shared preprocessing
For easy comparison across models, let's initially use the same one-hot encoded feature matrix.

In [22]:
def build_preprocessor(df, cols):

    numeric = [
        c for c in cols
        if pd.api.types.is_numeric_dtype(df[c])
    ]
    categorical = [
        c for c in cols
        if c not in numeric
    ]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(
                strategy="most_frequent"
            )),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=100
            ))
        ]), categorical)
    ])

4. Model factory
This is the key to making your pipeline flexible.
All model algorithms are defined in one function.


In [23]:
def make_model(task, algorithm, params):

    if task == "frequency":

        if algorithm == "glm":
            return PoissonRegressor(
                **params,
                max_iter=500
            )

        if algorithm == "tree":
            return DecisionTreeRegressor(
                criterion="poisson",
                random_state=SEED,
                **params
            )

        if algorithm == "xgboost":
            from xgboost import XGBRegressor

            return XGBRegressor(
                objective="count:poisson",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1,
                **params
            )

    elif task == "severity":

        if algorithm == "glm":
            return GammaRegressor(
                **params,
                max_iter=500
            )

        if algorithm == "tree":
            return DecisionTreeRegressor(
                criterion="poisson",
                random_state=SEED,
                **params
            )

        if algorithm == "xgboost":
            from xgboost import XGBRegressor

            return XGBRegressor(
                objective="reg:gamma",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1,
                **params
            )

    raise ValueError(f"Unknown combination: {task}, {algorithm}")



The main complication is exposure: scikit-learn's Poisson GLM and trees do not accept a dedicated offset, whereas XGBoost can incorporate a log-exposure base margin.
The following adapter hides that difference from the rest of the pipeline.

The first few lines of fit_component above can be simplified: we'll always pass severity-filtered inputs when fitting severity, so the only required check is that len(df) > 0.
This adapter means you can swap between Poisson GLM, a tree and XGBoost without changing how the final claim count is computed. Note that regularization and tree-splitting conventions can differ across implementations, so exposure equivalence refers to the underlying Poisson loss, not necessarily identical fitted models.

In [24]:
def fit_component(model, task, algorithm, X, df):

    if task == "severity":
        sev = severity_filter(df)
        raise_if_empty = len(sev) == 0
        if raise_if_empty:
            raise ValueError("No positive severity observations")

        # X and df must already be severity-filtered.
        model.fit(
            X,
            df[SEV_TARGET],
            sample_weight=df[FREQ_TARGET]
        )
        return model

    exposure = df[EXPOSURE].to_numpy()
    counts = df[FREQ_TARGET].to_numpy()

    if algorithm == "xgboost":
        # Explicit log-exposure offset via base margin.
        model.fit(
            X,
            counts,
            base_margin=np.log(exposure)
        )
    else:
        # Exposure-weighted Poisson rate formulation.
        # Equivalent to the count-offset Poisson loss,
        # up to terms independent of fitted rates.
        model.fit(
            X,
            counts / exposure,
            sample_weight=exposure
        )

    return model


def predict_component(
    model, task, algorithm, X, df
):
    if task == "severity":
        return np.maximum(model.predict(X), 1e-9)

    exposure = df[EXPOSURE].to_numpy()

    if algorithm == "xgboost":
        return model.predict(
            X,
            base_margin=np.log(exposure)
        )

    return exposure * model.predict(X)

6. One reusable fitting function
The function below can train any selected frequency/severity combination for one line.

In [25]:
def fit_pair(
    df,
    cols,
    freq_algorithm,
    sev_algorithm,
    freq_params,
    sev_params
):
    preprocessor = build_preprocessor(df, cols)

    X = preprocessor.fit_transform(df[cols])

    # Frequency: all rows
    freq_model = make_model(
        "frequency", freq_algorithm, freq_params
    )

    fit_component(
        freq_model, "frequency",
        freq_algorithm, X, df
    )


    # Severity: positive average claims
    mask = (
        (df[SEV_TARGET] > 0)
    ).to_numpy()

    df_sev = df.loc[mask].copy()

    # Cap severity at the 99.5th percentile
    sev_cap = df_sev[SEV_TARGET].quantile(CAP_QUANTILE)

    df_sev[SEV_TARGET] = df_sev[SEV_TARGET].clip(
        upper=sev_cap
    )

    sev_model = make_model(
        "severity", sev_algorithm, sev_params
    )

    fit_component(
        sev_model, "severity",
        sev_algorithm, X[mask], df_sev
    )

    return {
        "preprocessor": preprocessor,
        "frequency": freq_model,
        "severity": sev_model,
        "freq_algorithm": freq_algorithm,
        "sev_algorithm": sev_algorithm
    }


def predict_pair(bundle, df, cols):

    X = bundle["preprocessor"].transform(df[cols])

    expected_n = predict_component(
        bundle["frequency"],
        "frequency",
        bundle["freq_algorithm"],
        X, df
    )

    expected_sev = predict_component(
        bundle["severity"],
        "severity",
        bundle["sev_algorithm"],
        X, df
    )

    return pd.DataFrame({
        "row_id": df["row_id"].to_numpy(),
        "expected_n": expected_n,
        "expected_sev": expected_sev,
        "predicted_cost": expected_n * expected_sev
    }, index=df.index)

7. Define evaluation metrics
We evaluate each component and, most importantly, the combined expected claim cost.
For model selection, I suggest a fixed-power Tweedie deviance on annualized aggregate cost, using exposure weights. This allows fair comparison of different frequency–severity combinations.

In [26]:
from sklearn.metrics import (
    d2_tweedie_score,
    mean_tweedie_deviance,
    mean_absolute_error,
    mean_squared_error
)

def evaluate_pair(df, pred):

    actual = df["net_incurred"].to_numpy()
    predicted = pred["predicted_cost"].to_numpy()
    exposure = df["exposure"].to_numpy()

    return {
        "D2_tweedie": d2_tweedie_score(
            actual,
            predicted,
            power=1.5
        ),

        "cost_deviance": mean_tweedie_deviance(
            actual,
            predicted,
            power=1.5
        ),

        "MAE": mean_absolute_error(
            actual,
            predicted
        ),

        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),

        "actual_PP": actual.sum() / exposure.sum(),

        "predicted_PP": predicted.sum() / exposure.sum(),

        "A/E": actual.sum() / predicted.sum()
    }

8. Hyperparameter grid generation
This helper converts the parameter grids into candidate configurations.
You can now add algorithms or expand tuning ranges without changing the training code.

In [27]:
def parameter_grid(grid):
    keys = list(grid)

    for values in product(*(grid[k] for k in keys)):
        yield dict(zip(keys, values))


def model_configs(task):
    for algorithm in ACTIVE_MODELS[task]:
        for params in parameter_grid(
            MODEL_GRIDS[task][algorithm]
        ):
            yield algorithm, params

9. Hyperparameter tuning using 2023
Rather than evaluate every combination on 2024, tune on 2023 first.
The following function tests all selected frequency–severity combinations and ranks them using the combined cost deviance.

In [28]:
def tune_line(line):

    df = train_split[line]
    cols = features[line]

    inner_train = df.loc[
        df["year"] <= 2022
    ].copy()

    inner_valid = df.loc[
        df["year"] == 2023
    ].copy()

    results = []

    for freq_algo, freq_params in model_configs("frequency"):
        for sev_algo, sev_params in model_configs("severity"):

            print(
                line, freq_algo, freq_params,
                "|", sev_algo, sev_params
            )

            bundle = fit_pair(
                inner_train,
                cols,
                freq_algo,
                sev_algo,
                freq_params,
                sev_params
            )

            pred = predict_pair(
                bundle, inner_valid, cols
            )

            metrics = evaluate_pair(
                inner_valid, pred
            )

            results.append({
                "line": line,
                "freq_algorithm": freq_algo,
                "sev_algorithm": sev_algo,
                "freq_params": freq_params,
                "sev_params": sev_params,
                **metrics
            })

    return pd.DataFrame(results).sort_values(
        "D2_tweedie", ascending=False
    ).reset_index(drop=True)


tuning_results = {
    line: tune_line(line)
    for line in LINES
}


# Run the search:
for line, result in tuning_results.items():
    print(f"\n=== {line.upper()} ===")
    display(result.head(10))

motor xgboost {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8} | xgboost {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}
motor xgboost {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8} | xgboost {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}
motor xgboost {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8} | xgboost {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}
motor xgboost {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8} | xgboost {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}
buildings xgboost {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8} | xgboost {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}
buildings xgboost {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsam

,line,freq_algorithm,sev_algorithm,freq_params,sev_params,D2_tweedie,cost_deviance,MAE,RMSE,actual_PP,predicted_PP,A/E
0,motor,xgboost,xgboost,"{'max_depth': 5, 'learning_rate': 0.05, 'n_est...","{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",0.074766,121.529308,747.414994,23814.195574,588.324162,482.488954,1.219353
1,motor,xgboost,xgboost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...","{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",0.073568,121.686674,750.401095,23814.295548,588.324162,484.107409,1.215276
2,motor,xgboost,xgboost,"{'max_depth': 5, 'learning_rate': 0.05, 'n_est...","{'max_depth': 5, 'learning_rate': 0.05, 'n_est...",0.064954,122.818225,721.842163,23814.423172,588.324162,440.992945,1.334090
3,motor,xgboost,xgboost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...","{'max_depth': 5, 'learning_rate': 0.05, 'n_est...",0.063891,122.957823,724.649035,23814.517917,588.324162,442.508636,1.329520



=== BUILDINGS ===


,line,freq_algorithm,sev_algorithm,freq_params,sev_params,D2_tweedie,cost_deviance,MAE,RMSE,actual_PP,predicted_PP,A/E
0,buildings,xgboost,xgboost,"{'max_depth': 5, 'learning_rate': 0.05, 'n_est...","{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",0.090613,116.958500,593.124296,4609.127681,432.459334,328.230359,1.317548
1,buildings,xgboost,xgboost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...","{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",0.089328,117.123873,592.041425,4610.373991,432.459334,325.806527,1.327350
2,buildings,xgboost,xgboost,"{'max_depth': 5, 'learning_rate': 0.05, 'n_est...","{'max_depth': 5, 'learning_rate': 0.05, 'n_est...",0.044074,122.944001,536.772236,4613.329313,432.459334,251.689743,1.718224
3,buildings,xgboost,xgboost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...","{'max_depth': 5, 'learning_rate': 0.05, 'n_est...",0.042348,123.166006,536.117252,4614.499885,432.459334,250.051258,1.729483



=== CONTENTS ===


,line,freq_algorithm,sev_algorithm,freq_params,sev_params,D2_tweedie,cost_deviance,MAE,RMSE,actual_PP,predicted_PP,A/E
0,contents,xgboost,xgboost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...","{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",0.072696,61.429350,163.034285,1038.462527,117.070254,94.916392,1.233404
1,contents,xgboost,xgboost,"{'max_depth': 5, 'learning_rate': 0.05, 'n_est...","{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",0.072505,61.441969,163.820937,1038.198607,117.070254,96.185691,1.217128
2,contents,xgboost,xgboost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...","{'max_depth': 5, 'learning_rate': 0.05, 'n_est...",0.036737,63.811465,149.718396,1039.708279,117.070254,76.715879,1.526024
3,contents,xgboost,xgboost,"{'max_depth': 5, 'learning_rate': 0.05, 'n_est...","{'max_depth': 5, 'learning_rate': 0.05, 'n_est...",0.036028,63.858406,150.312389,1039.476715,117.070254,77.681894,1.507047


10. Train the selected models on 2021–2023
Now select the top-ranked configuration for each line and refit it using all observations in your existing train_split.

In [29]:
best_configs = {}
models = {}

for line in LINES:

    best = tuning_results[line].iloc[0]

    best_configs[line] = {
        "freq_algorithm": best["freq_algorithm"],
        "sev_algorithm": best["sev_algorithm"],
        "freq_params": best["freq_params"],
        "sev_params": best["sev_params"]
    }

    models[line] = fit_pair(
        train_split[line],
        features[line],
        **best_configs[line]
    )

    print(line, best_configs[line])

motor {'freq_algorithm': 'xgboost', 'sev_algorithm': 'xgboost', 'freq_params': {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}, 'sev_params': {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}}
buildings {'freq_algorithm': 'xgboost', 'sev_algorithm': 'xgboost', 'freq_params': {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}, 'sev_params': {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}}
contents {'freq_algorithm': 'xgboost', 'sev_algorithm': 'xgboost', 'freq_params': {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}, 'sev_params': {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300, 'subsample': 0.8}}


11. Evaluate on untouched 2024 validation data
This is your primary out-of-time evaluation.

In [30]:

# Generate fresh predictions from the fitted models
valid_predictions = {
    line: predict_pair(
        models[line],
        valid_split[line],
        features[line]
    )
    for line in LINES
}

# Check row alignment
for line in LINES:
    assert np.array_equal(
        valid_split[line]["row_id"].to_numpy(),
        valid_predictions[line]["row_id"].to_numpy()
    )


# Evaluate individual lines
validation_results = []

for line in LINES:
    metrics = evaluate_pair(
        valid_split[line],
        valid_predictions[line]
    )

    validation_results.append({
        "line": line,
        **metrics
    })

# Combine actual and predicted costs across all lines
all_actual = pd.concat(
    [valid_split[line] for line in LINES],
    ignore_index=True
)

all_predictions = pd.concat(
    [valid_predictions[line] for line in LINES],
    ignore_index=True
)

# Evaluate entire portfolio
total_metrics = evaluate_pair(
    all_actual,
    all_predictions
)

validation_results.append({
    "line": "TOTAL",
    **total_metrics
})

validation_results = (
    pd.DataFrame(validation_results)
    .set_index("line")
)

display(validation_results.round(3))

,D2_tweedie,cost_deviance,MAE,RMSE,actual_PP,predicted_PP,A/E
line,,,,,,,
motor,0.080,104.892,646.146,2870.681,455.390,481.723,0.945
buildings,0.102,118.205,633.706,5004.104,451.290,362.713,1.244
contents,0.079,61.107,166.120,1033.564,117.233,99.040,1.184
TOTAL,0.125,95.694,498.763,3321.122,352.668,331.161,1.065


In [31]:
final_models = {
    line: fit_pair(
        train[line],
        features[line],
        **best_configs[line]
    )
    for line in LINES
}

submission = pd.concat(
    [
        predict_pair(
            final_models[line],
            test[line],
            features[line]
        )[["row_id", "predicted_cost"]]
        for line in LINES
    ],
    ignore_index=True
)

submission = submission.rename(
    columns={"predicted_cost": "prediction"}
)

sample = datasets["sample_submission"][["row_id"]].copy()

assert submission["row_id"].is_unique
assert set(submission["row_id"]) == set(sample["row_id"])

submission = sample.merge(
    submission,
    on="row_id",
    how="left",
    validate="one_to_one"
)

assert submission["prediction"].notna().all()
assert np.isfinite(submission["prediction"]).all()
assert (submission["prediction"] >= 0).all()

submission.to_csv(
    "submission_freq_sev.csv",
    index=False
)

In [32]:
# validation vs test prediction distribution, per line
for line in LINES:
    pv = valid_predictions[line].predicted_cost
    pt = predict_pair(final_models[line], test[line], features[line]).predicted_cost
    print(line)
    print("  valid PP:", pv.sum()/valid_split[line].exposure.sum(),
          "| test PP:", pt.sum()/test[line].exposure.sum())
    print("  valid q99/max:", pv.quantile(.99), pv.max())
    print("  test  q99/max:", pt.quantile(.99), pt.max())

motor
  valid PP: 481.7231071852288 | test PP: 458.14641193762657
  valid q99/max: 1355.641710205078 5405.835
  test  q99/max: 1301.3549560546874 5848.8027
buildings
  valid PP: 362.7134849817697 | test PP: 395.2186436575048
  valid q99/max: 1528.3944091796875 27056.467
  test  q99/max: 1639.5198876953077 15311.629
contents
  valid PP: 99.03961770963353 | test PP: 106.12076298468317
  valid q99/max: 343.7165802001955 1945.9948
  test  q99/max: 344.7911770629882 1836.968
